<a href="https://colab.research.google.com/github/jujurdewa/data-science-2026/blob/main/Pertemuan12_Jujur_Dewa_Pamungkas_250401020107.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama : Jujur Dewa Pamungkas

Nim : 250401020107

Prodi : PJJ Informatika

kelas : IF403

In [10]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk  = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
           'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

transaksi = []
for _ in range (50):
    n_item  =  np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
      transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())


    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [11]:
import warnings
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')
freq_items  = apriori(df, min_support=0.1, use_colnames=True)
freq_items  = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))


min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [13]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("\n--- Aturan Asosiasi Terkuat ---")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


--- Aturan Asosiasi Terkuat ---
         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
14  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
10     (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
15   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


**Aturan mana yang paling kuat (Lift tertinggi)?**
Berdasarkan simulasi penyuntikan pola di awal, aturan asosiasi dengan Lift tertinggi akan melibatkan pasangan (Roti) -> (Selai) atau sebaliknya. Nilai Lift > 1 menunjukkan bahwa pembelian Roti memiliki korelasi positif yang sangat kuat terhadap pembelian Selai.

**Apakah masuk akal secara bisnis?**
Sangat masuk akal. Secara logika bisnis dan kebiasaan konsumen, Roti dan Selai adalah produk komplementer (saling melengkapi). Orang yang membeli roti tawar biasanya akan membeli selai sebagai pasangannya. Temuan ini bisa dimanfaatkan swalayan untuk menempatkan rak Roti bersebelahan dengan rak Selai.

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('\nMirip dengan Roti:', rekomendasi_serupa('Roti'))


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [15]:
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('\n--- Perbandingan Rekomendasi ---')
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('\nRekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))


--- Perbandingan Rekomendasi ---
Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


**Apakah kedua pendekatan memberi rekomendasi yang konsisten?**
Hasilnya bisa tumpang tindih (misalnya keduanya merekomendasikan "Selai"), namun konsep dasar mereka berbeda.

Content-Based merekomendasikan produk substitusi atau produk di rak yang sama (merekomendasikan 'Selai' dan 'Sereal' karena sama-sama kategori 'Bakery').

Association Rules merekomendasikan produk berdasarkan perilaku nyata pembeli (apa yang sering masuk ke keranjang bersamaan), meskipun kategorinya berbeda (misalnya Roti dengan Susu).

**Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?**

Gunakan Content-Based: Saat toko memiliki produk baru yang belum pernah dibeli sama sekali (Cold Start Problem). Model tetap bisa merekomendasikan produk ini karena model hanya melihat atribut/kategori produk.

Gunakan Association Rules: Saat sistem sudah memiliki banyak data transaksi historis. Sangat cocok untuk promosi cross-selling (bundling produk yang saling melengkapi).

Gunakan Hybrid: Ini adalah pendekatan terbaik (digunakan oleh e-commerce besar). Gabungan ini memungkinkan sistem merekomendasikan produk komplementer (berdasarkan transaksi) sekaligus menangani produk-produk baru yang belum laku terjual (berdasarkan konten/kategori).

**KESIMPULAN**

Berdasarkan seluruh rangkaian proses pembelajaran dan kode yang telah kita bahas dari awal hingga akhir, kesimpulannya adalah kamu telah berhasil mempraktikkan Tiga Pilar Utama dalam Data Science untuk Bisnis (Customer Analytics):

**1. Analisis Prediktif (Supervised Learning) — Prediksi Churn Pelanggan**

Kamu belajar cara memprediksi apakah seorang pelanggan akan berhenti berlangganan (churn) atau tidak menggunakan algoritma Random Forest.

Kesimpulan Bisnis: Menangani data yang tidak seimbang (imbalanced data) sangat penting (misalnya dengan class_weight="balanced") agar model tidak bias. Fokus utama di sini adalah meminimalkan False Negatives (meningkatkan Recall) agar perusahaan tidak kecolongan pelanggan yang berisiko pergi.

**2. Segmentasi Pelanggan (Unsupervised Learning) — Clustering**

Kamu berhasil mengelompokkan pelanggan berdasarkan kesamaan perilaku (Pendapatan dan Skor Belanja).

Kesimpulan Analitis: Baik menggunakan pendekatan K-Means (Elbow Method) maupun Hierarchical Clustering (Dendrogram), keduanya secara konsisten menunjukkan bahwa pelanggan paling optimal dibagi menjadi 3 segmen: kelompok Hemat, Menengah, dan Boros/Premium. Ini memungkinkan perusahaan untuk membuat strategi promosi yang lebih tepat sasaran untuk masing-masing kelompok.

**3. Sistem Rekomendasi (Association & Content-Based)**

Kamu mempraktikkan cara mencari pola pembelian barang (Market Basket Analysis dengan Apriori) dan merekomendasikan produk berdasarkan kemiripan kategori (Content-Based Filtering).

Kesimpulan Bisnis: Aturan asosiasi menemukan pola perilaku nyata (misal: Roti sering dibeli bersama Selai), yang sangat bagus untuk strategi cross-selling (penempatan barang berdekatan/diskon bundling). Sementara itu, Content-Based sangat berguna untuk merekomendasikan produk baru yang belum pernah dibeli. Pendekatan hibrida (menggabungkan keduanya) adalah solusi paling ideal di dunia nyata.